In [15]:
from pyftdi.usbtools import UsbTools

devs = UsbTools.find_all([(0x0403, 0x6014), (0x0403, 0x6010)])
if not devs:
    raise RuntimeError("No FTDI device found")

desc, iface = devs[0]  # <- unpack the tuple
url = f"ftdi://{desc.vid:04x}:{desc.pid:04x}:{desc.bus}:{desc.address}/{iface}"
print("Using:", url)

Using: ftdi://0403:6014:1:4/1


In [18]:
# Fast GPIO toggle on FT232H using pyftdi (bitbang via GpioController)
# Exit with Ctrl+C

import sys
import time
from pyftdi.gpio import GpioController
from pyftdi.usbtools import UsbTools

# --- Config ---
FTDI_URL = "ftdi://ftdi:232h/1"
FTDI_URL = "ftdi://::/1" 
PIN = 7                 # ADBUS7 = pin 7 (D7). Change to 1 for D1, etc.
MASK = 1 << PIN
TOGGLE_BATCH = 1024     # how many high/low toggles per "batch" write loop
PRINT_EVERY_S = 1.0     # stats interval

gpio = GpioController()
gpio.open_from_url(FTDI_URL)

# Set the pin as output, initialize low
gpio.set_direction(MASK, MASK)
gpio.write(0)

# Precompute alternating states for faster loop (still USB-limited)
# We'll flip between 0 and MASK
state0 = 0
state1 = MASK

print(f"Toggling ADBUS{PIN} (mask 0x{MASK:02x}). Ctrl+C to stop...")

t0 = time.perf_counter()
last_print = t0
toggles = 0

try:
    while True:
        # Do a bunch of toggles per Python loop to reduce overhead
        for _ in range(TOGGLE_BATCH):
            gpio.write(state1)
            gpio.write(state0)
        toggles += 2 * TOGGLE_BATCH  # each write is one edge-state update

        now = time.perf_counter()
        if now - last_print >= PRINT_EVERY_S:
            dt = now - t0
            rate = toggles / dt  # "state updates" per second
            # If you're measuring frequency of a square wave, f ~= (rate / 2)
            # print(f"{dt:8.2f}s  state-updates: {toggles:12d}  rate: {rate:,.0f}/s  ~squarewave: {rate/2:,.0f} Hz")
            sys.stdout.write(f"\r{dt:8.2f}s  state-updates: {toggles:12d}  rate: {rate:,.0f}/s  ~squarewave: {rate/2:,.0f} Hz")
            sys.stdout.flush()
            last_print = now

except KeyboardInterrupt:
    print("\nStopping...")

finally:
    # leave pin low and close cleanly
    try:
        gpio.write(0)
    except Exception:
        pass
    gpio.close()
    print("Closed FTDI GPIO.")


Toggling ADBUS7 (mask 0x80). Ctrl+C to stop...
   49.25s  state-updates:       753664  rate: 15,302/s  ~squarewave: 7,651 Hz
Stopping...
Closed FTDI GPIO.
